## CODE TO SCRAPE ALL ~340 FILES
AND 
## CODE TO SELECT ONLY 15 OF VALID FILES SCRAPED

In [ ]:
import pandas as pd
import requests
import json
import time
from bs4 import BeautifulSoup

df = pd.read_csv("final_gold_standard.csv")

def scrape_pmc(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    
    if "pubmed.ncbi.nlm.nih.gov" in url:
        pubmed_id = url.strip("/").split("/")[-1]
        search_url = f"https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/?ids={pubmed_id}&format=json"
        try:
            r = requests.get(search_url, headers=headers)
            data = r.json()
            pmc_id = data["records"][0].get("pmcid")
            if pmc_id:
                url = f"https://pmc.ncbi.nlm.nih.gov/articles/{pmc_id}/"
        except:
            pass

    try:
        r = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")

        for tag in soup.find_all(["table", "figure", "sup"]):
            tag.decompose()

        article = soup.find("div", class_="article-body") or \
                  soup.find("div", id="body") or \
                  soup.find("main") or \
                  soup.find("article")

        if article:
            return article.get_text(separator=" ", strip=True)
        else:
            return soup.get_text(separator=" ", strip=True)[:50000]

    except Exception as e:
        return f"ERROR: {e}"

results = []
failed = []

for _, row in df.iterrows():
    title = row["Title"]
    link = str(row["Link (Use DOI or Title if missing)"])
    print(f"Scraping: {title[:60]}...")

    if link == "nan" or not link.startswith("http"):
        print("  No link, skipping")
        continue

    text = scrape_pmc(link)
    time.sleep(2)

    char_len = len(text)
    is_usable = char_len > 5000 and "Enable JavaScript" not in text and "Just a moment" not in text

    print(f"  {'OK' if is_usable else 'FAILED'} — {char_len} chars")

    entry = {
        "title": title,
        "link": link,
        "disease": row["Disease"],
        "taxa_enriched": row["KeyTaxa_Enriched"],
        "taxa_depleted": row["KeyTaxa_Depleted"],
        "in_gold_standard": row["InGoldStandard"],
        "char_len": char_len,
        "usable": is_usable,
        "text": text
    }

    if is_usable:
        results.append(entry)
    else:
        failed.append(title)

print(f"\nUsable: {len(results)} / {len(df)}")
print(f"Failed: {len(failed)}")

# Save all usable
with open("all_usable_papers.json", "w") as f:
    json.dump(results, f, indent=2)

# Sample 10 yes 5 no from usable
usable_df = pd.DataFrame([{
    "title": r["title"],
    "in_gold_standard": r["in_gold_standard"]
} for r in results])

yes_pool = usable_df[usable_df["in_gold_standard"] == "Yes"]
no_pool = usable_df[usable_df["in_gold_standard"] == "No"]

print(f"\nUsable Yes: {len(yes_pool)}, Usable No: {len(no_pool)}")

yes_sample = yes_pool.sample(n=min(10, len(yes_pool)), random_state=42)
no_sample = no_pool.sample(n=min(5, len(no_pool)), random_state=42)

selected_titles = set(yes_sample["title"].tolist() + no_sample["title"].tolist())
final_15 = [r for r in results if r["title"] in selected_titles]

with open("gold_standard_final_15.json", "w") as f:
    json.dump(final_15, f, indent=2)

print(f"Saved {len(final_15)} papers to gold_standard_final_15.json")

Scraping: Dysbiosis characteristics of gut microbiota in cerebral infa...
  OK — 47404 chars
Scraping: Gut Microbiota and Fecal Metabolites Associated With Neuroco...
  OK — 41253 chars
Scraping: Hemorrhagic transformation in patients with large-artery ath...
  FAILED — 58 chars
Scraping: Rett Syndrome: A Focus on Gut Microbiota...
  FAILED — 224 chars
Scraping: The Alterations of Gut Microbiome and Lipid Metabolism in Pa...
  OK — 42573 chars
Scraping: Distinctive Gut Microbiota Alteration Is Associated with Pos...
  FAILED — 58 chars
Scraping: Comparison of the effects of probiotics, rifaximin, and lact...
  OK — 47747 chars
Scraping: Oral Pathobiont Streptococcus Anginosus Is Enriched in the G...
  OK — 8366 chars
Scraping: Integrated Traditional Chinese Medicine Improves Functional ...
  OK — 52344 chars
Scraping: The gut microbial signatures of patients with lacunar cerebr...
  FAILED — 58 chars
Scraping: Altered Gut Microbiota and Plasma Metabolome Profiles Charac...
  OK — 50000

### QUICK VERIFICATION

In [10]:
import json

with open("gold_standard_final_15.json", "r") as f:
    papers = json.load(f)

for p in papers:
    print(f"[{p['in_gold_standard']}] {p['title'][:70]} — {p['char_len']} chars")

[No] Intestinal flora induces depression by mediating the dysregulation of  — 59748 chars
[Yes] Gut microbes exacerbate systemic inflammation and behavior disorders i — 93335 chars
[Yes] Alterations in gut microbiota and metabolomic profiles in acute stroke — 67570 chars
[No] Gut microbiome dysbiosis across early Parkinson's disease, REM sleep b — 79012 chars
[Yes] The gut microbiota in multiple sclerosis varies with disease activity. — 80834 chars
[No] Gut Microbial Ecosystem in Parkinson Disease: New Clinicobiological In — 9977 chars
[Yes] Dysbiosis of gut microbiota in a selected population of Parkinson's pa — 5985 chars
[No] Gut microbiota distinguishes aging hispanics with Alzheimer's disease: — 75825 chars
[No] Examining the complex Interplay between gut microbiota abundance and s — 49363 chars
[Yes] Characterizing Gut Microbiota in Older Chinese Adults with Cognitive I — 6996 chars
[Yes] Gut microbiome, cognitive function and brain structure: a multi-omics  — 64883 chars
[Yes] R